# micrograd exercises

1. watch the [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) on YouTube
2. come back and complete these exercises to level up :)

## section 1: derivatives

In [1]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [2]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

def gradf(a, b, c):
  da = -3*a**2-0.5*a**-.5
  db = 3*cos(3*b)+2.5*b**1.5
  dc = 1/c**2
  return [da, db, dc] # todo, return [df/da, df/db, df/dc]

# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [3]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell
a=2; b=3; c=4;

# -----------
epsilon = 0.000001
deltaA = (f(a+epsilon, b, c) - f(a, b, c))/epsilon
deltaB = (f(a, b+epsilon, c) - f(a, b, c))/epsilon
deltaC = (f(a, b, c+epsilon) - f(a, b, c))/epsilon
numerical_grad = [deltaA, deltaB, deltaC] # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353559348809995
OK for dim 1: expected 10.25699027111255, yours returns 10.256991666679482
OK for dim 2: expected 0.0625, yours returns 0.062499984743169534


In [4]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.

# -----------
delta2A = (f(a+epsilon, b, c) - f(a-epsilon, b, c))/(2*epsilon)
delta2B = (f(a, b+epsilon, c) - f(a, b-epsilon, c))/(2*epsilon)
delta2C = (f(a, b, c+epsilon) - f(a, b, c-epsilon))/(2*epsilon)
numerical_grad2 = [delta2A, delta2B, delta2C] # TODO
# -----------

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")

diffs = [x-y for x,y in zip(numerical_grad, numerical_grad2)]
for dim in range(len(diffs)):
  print(f"Original approximation for dim {dim}: {diffs[dim]}");

OK for dim 0: expected -12.353553390593273, yours returns -12.353553391353245
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027401572
OK for dim 2: expected 0.0625, yours returns 0.06250000028629188
Original approximation for dim 0: -5.95745675013859e-06
Original approximation for dim 1: 1.3926637620897964e-06
Original approximation for dim 2: -1.554312234475219e-08


## section 2: support for softmax

In [5]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  def __radd__(self, other):
        return self+other

  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here
  def exp(self):
    out = Value(exp(self.data), (self, ), 'exp')

    def _backward():
      self.grad += out.data*out.grad
    out._backward = _backward

    return out

  def log(self):
    out = Value(log(self.data), (self, ), 'ln')

    def _backward():
      self.grad += (1.0/self.data)*out.grad
    out._backward = _backward

    return out

  def __truediv__(self, other):
    other = other if isinstance (other, Value) else Value(other)
    out = Value(self.data / other.data, (self, other), '/')

    def _backward():
      self.grad  += (1.0/other.data)*out.grad
      other.grad += -self.data * (1.0/other.data**2)*out.grad
    out._backward = _backward

    return out

  def __neg__(self):
    out = Value(-1.0*self.data, (self, ), 'n')

    def _backward():
      self.grad += -1.0*out.grad
    out._backward = _backward

    return out
  # ------

  def backward(self): # exactly as in video
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [6]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  counts = [logit.exp() for logit in logits]
  denominator = sum(counts)
  out = [c / denominator for c in counts]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


2.1755153626167147
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.886450380640099


In [8]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch

l1 = torch.Tensor([ 0.0]).double(); l1.requires_grad = True
l2 = torch.Tensor([ 3.0]).double(); l2.requires_grad = True
l3 = torch.Tensor([-2.0]).double(); l3.requires_grad = True
l4 = torch.Tensor([ 1.0]).double(); l4.requires_grad = True

tlogits = [l1, l2, l3, l4]
probs = softmax(tlogits)
tloss = -probs[3].log()
tloss.backward()

print('l1', l1.grad.item())
print('l2', l2.grad.item())
print('l3', l3.grad.item())
print('l4', l4.grad.item())

l1 0.041772570515350445
l2 0.8390245074625319
l3 0.005653302662216329
l4 -0.8864503806400988
